# dazpy â€” Interactive Workspace

Connect Python to a live DAZ Studio session and explore the scene, control figures, and trigger renders.

## Prerequisites

- DAZ Studio is running with the **DazScriptServer** plugin active
- Default address: `http://127.0.0.1:18811`
- Verify with: `curl http://127.0.0.1:18811/health`

## API Documentation

| Section | Link |
|---|---|
| Overview | [bluemoonfoundry.github.io/daz-script-server/](https://bluemoonfoundry.github.io/daz-script-server/) |
| Quick Start | [quickstart](https://bluemoonfoundry.github.io/daz-script-server/quickstart.html) |
| DazClient | [api/client](https://bluemoonfoundry.github.io/daz-script-server/api/client.html) |
| DazScene | [api/scene](https://bluemoonfoundry.github.io/daz-script-server/api/scene.html) |
| DazSkeleton / DazBone | [api/skeleton](https://bluemoonfoundry.github.io/daz-script-server/api/skeleton.html) |
| DazNode | [api/nodes](https://bluemoonfoundry.github.io/daz-script-server/api/nodes.html) |
| DazCamera | [api/camera](https://bluemoonfoundry.github.io/daz-script-server/api/camera.html) |
| DazLight | [api/light](https://bluemoonfoundry.github.io/daz-script-server/api/light.html) |
| DazMaterial | [api/materials](https://bluemoonfoundry.github.io/daz-script-server/api/materials.html) |
| DazRenderSettings | [api/render](https://bluemoonfoundry.github.io/daz-script-server/api/render.html) |
| DazTimeline / DazAnimation | [api/timeline](https://bluemoonfoundry.github.io/daz-script-server/api/timeline.html) |
| DazPose | [api/pose](https://bluemoonfoundry.github.io/daz-script-server/api/pose.html) |
| DazGeometry | [api/geometry](https://bluemoonfoundry.github.io/daz-script-server/api/geometry.html) |
| DazProperty | [api/properties](https://bluemoonfoundry.github.io/daz-script-server/api/properties.html) |
| ExecutionResult | [api/result](https://bluemoonfoundry.github.io/daz-script-server/api/result.html) |
| Exceptions | [api/exceptions](https://bluemoonfoundry.github.io/daz-script-server/api/exceptions.html) |
| Vec3 / Quat / BoundingBox | [api/math3](https://bluemoonfoundry.github.io/daz-script-server/api/math3.html) |
| Batch | [api/batch](https://bluemoonfoundry.github.io/daz-script-server/api/batch.html) |

## Quick Reference

### Key classes

| Class | Import | Purpose |
|---|---|---|
| `DazClient` | `dazpy` | Low-level HTTP client â€” `execute()`, `health()`, `metrics()` |
| `DazScene` | `dazpy` | Scene-level queries â€” nodes, skeletons, cameras, frame, undo |
| `DazSkeleton` | `dazpy` | Figure with bones â€” `find_bone()`, `bones()`, `apply_pose()` |
| `DazBone` | `dazpy` | Individual bone â€” `set_local_rotation()`, `local_euler` |
| `DazNode` | `dazpy` | Generic scene node â€” `label`, `position`, `visible` |
| `DazCamera` | `dazpy` | Camera node â€” `focal_length`, `set_active()` |
| `DazLight` | `dazpy` | Light node â€” `intensity`, `color` |
| `DazMaterial` | `dazpy` | Material on a node â€” `diffuse_color`, `set_property()` |
| `DazRenderSettings` | `dazpy` | Active render settings â€” `width`, `height`, `engine` |
| `DazTimeline` | `dazpy` | Playback range, FPS, current frame |
| `DazPose` | `dazpy` | Captured pose snapshot â€” save / restore bone rotations |
| `DazGeometry` | `dazpy` | Mesh data â€” vertex count, face count, UVs |
| `Batch` | `dazpy` | Fan-out multiple operations in fewer HTTP round-trips |

### DazScript rules (inline scripts via `client.execute()`)

- **No top-level `return`** â€” the result is the value of the last expression: `App.version;`
- **Multi-statement scripts need an IIFE**: wrap in `(function() { ...; return value; })()`  
  `ScriptBuilder.iife(code)` does this for you when working with the SDK
- **Result lives in `.value`**, not `.result`: `r = client.execute(script); r.value`
- **Print via `print()`** in DazScript; output appears in `result.output` (list of strings)

### Example scripts

| Topic | Path |
|---|---|
| Raw script / primary selection | `docs/examples/fundamentals/raw_script.py` |
| Full scene inventory (JSON) | `docs/examples/fundamentals/scene_introspection.py` |
| Scene node inventory | `docs/examples/fundamentals/scene_inventory.py` |
| Save scene copy | `docs/examples/fundamentals/scene_save_copy.py` |
| Pose transfer between figures | `docs/examples/character/pose_transfer.py` |
| Character state dump | `docs/examples/character/character_state.py` |
| Batch render morph variations | `docs/examples/rendering/batch_render_morph_variations.py` |
| Turntable render | `docs/examples/rendering/turntable.py` |
| Multi-camera render | `docs/examples/rendering/multi_camera_render.py` |
| Body measurements | `docs/examples/geometry/body_measurements.py` |
| BVH motion import | `docs/examples/bvh/bvh_import.py` |
| USD export | `docs/examples/export/scene_to_usd.py` |

In [8]:
import dazpy
from dazpy import DazClient, DazScene

client = DazClient()  # default: 127.0.0.1:18811
print(f"dazpy {dazpy.__version__} â€” connected to {client._base}")

dazpy 2.6.0 â€” connected to http://127.0.0.1:18811


---
## 1. Server health

In [9]:
import pprint
pprint.pprint(client.health())

{'active_requests': 0,
 'auth_enabled': False,
 'running': True,
 'status': 'ok',
 'uptime_seconds': 14232,
 'version': '2.6.0'}


---
## 2. Scene overview

In [10]:
scene = DazScene(client)
print(f"{scene.num_nodes()} nodes  |  {scene.num_skeletons()} skeletons  |  frame {scene.frame()}")
print()
for n in scene.nodes():
    print(f"  {type(n).__name__:<14}  {n.name}")

502 nodes  |  11 skeletons  |  frame 0

  DazNode         Tonemapper Options
  DazNode         Environment Options
  DazSkeleton     Genesis9
  DazNode         hip
  DazNode         pelvis
  DazNode         l_thigh
  DazNode         l_shin
  DazNode         l_foot
  DazNode         l_toes
  DazNode         l_bigtoe1
  DazNode         l_bigtoe2
  DazNode         l_indextoe1
  DazNode         l_indextoe2
  DazNode         l_midtoe1
  DazNode         l_midtoe2
  DazNode         l_ringtoe1
  DazNode         l_ringtoe2
  DazNode         l_pinkytoe1
  DazNode         l_pinkytoe2
  DazNode         l_metatarsal
  DazNode         l_thightwist1
  DazNode         l_thightwist2
  DazNode         r_thigh
  DazNode         r_shin
  DazNode         r_foot
  DazNode         r_toes
  DazNode         r_bigtoe1
  DazNode         r_bigtoe2
  DazNode         r_indextoe1
  DazNode         r_indextoe2
  DazNode         r_midtoe1
  DazNode         r_midtoe2
  DazNode         r_ringtoe1
  DazNode         r_rin

---
## 3. Find a figure and read a bone

`scene.find_skeleton_by_label()` searches by the name shown in DAZ Studio's Scene panel.  
Adjust the label to match a figure in your scene.

In [12]:
figure = scene.find_skeleton_by_label("Agnes Montenegro")  # change to match your scene
bone = figure.find_bone("head")
print("head local euler (xyzÂ°):", bone.local_euler)

head local euler (xyzÂ°): (-3.49952101707458, -4.64459180831909, -0.12897689640522)


---
## 4. Raw DazScript

Use `client.execute()` to run arbitrary DazScript when the SDK doesn't cover what you need.

**Rules:**
- Single expression â†’ no wrapper needed, just end with `;`
- Multiple statements â†’ wrap in an IIFE: `(function() { ...; return value; })()`
- Result is in `result.value`; console output (`print()` in DazScript) is in `result.output`

In [ ]:
# Single expression â€” no IIFE needed
r = client.execute("App.version;")
print("DAZ Studio version (packed int):", r.value)

In [ ]:
# Multiple statements â€” IIFE required
r = client.execute("""
(function() {
    var node = Scene.getPrimarySelection();
    if (!node) return null;
    return {
        name:  node.getName(),
        label: node.getLabel(),
        type:  node.className()
    };
})()
""")
pprint.pprint(r.value)

---
## 5. Error handling

### Exception hierarchy

| Exception | When raised |
|---|---|
| `DazError` | Base class — catch this to handle any dazpy error |
| `ConnectionError` | Cannot reach the server (DAZ Studio not running, wrong port) |
| `AuthenticationError` | HTTP 401/403 — bad or missing API token, or IP blocked |
| `ScriptError` | Base class for errors originating inside DazScript execution |
| `ScriptSyntaxError` | DazScript parse/syntax error (typo, missing bracket) |
| `ScriptRuntimeError` | DazScript failed at runtime (TypeError, ReferenceError, etc.) |
| `NodeNotFoundError` | Requested scene node, bone, or skeleton not found |
| `TimeoutError` | HTTP request or async poll exceeded its timeout |
| `AsyncExecutionError` | Async job failed, was cancelled, or timed out while polling |
| `RenderError` | Render job failed on the DAZ Studio side |

`ScriptError` exposes `.diagnostic` — a formatted string with line-numbered source and `request_id` for log correlation.

In [ ]:
from dazpy.exceptions import (
    ConnectionError, AuthenticationError,
    ScriptSyntaxError, ScriptRuntimeError,
    NodeNotFoundError, TimeoutError, DazError,
)

try:
    r = client.execute("Scene.getPrimarySelection().getName();")
    print("selected node:", r.value)
except ConnectionError:
    print("DAZ Studio is not running or the server address is wrong.")
except AuthenticationError:
    print("API token rejected — check DazScriptServer settings.")
except ScriptSyntaxError as e:
    print("Syntax error in script:")
    print(e.diagnostic)
except ScriptRuntimeError as e:
    print("Runtime error:", e)
    print("request_id:", e.request_id)
except NodeNotFoundError as e:
    print("Node not found:", e)
except DazError as e:
    print("Unexpected dazpy error:", e)


---
## 6. API browser

Discover what methods and properties any dazpy object exposes — without leaving the notebook.

`dir(obj)` returns everything, including Python internals. The one-liner below filters those out so you only see the public API surface.

In [ ]:
# Swap in any dazpy object to browse its public API
obj = scene  # try: client, scene, figure, bone, ...

[m for m in dir(obj) if not m.startswith('_')]

---
## 7. Jupyter introspection

Jupyter has built-in help that makes the SDK self-documenting:

| Syntax | What you get |
|---|---|
| `obj?` | Docstring, signature, and type |
| `obj??` | Full Python source (when available) |
| `obj.method?` | Docstring for a specific method |

Run any of these in a cell on their own — no `print()` needed.

> **Tip:** works on any dazpy object: `client?`, `scene?`, `figure?`, `bone?`, `DazScene?`

In [ ]:
# Run each line in its own cell to get inline help:
# scene?         # docstring + signature
# scene??        # full Python source
# scene.nodes?   # method docstring
scene?

---
## 8. Interactive bone rotator (ipywidgets)

Use `ipywidgets` sliders to drive a bone in real time — DAZ Studio responds on every drag.

Install once if needed:
```
pip install ipywidgets
```

**How it works:** Each slider’s `observe()` callback calls `bone.set_local_rotation(x, y, z)`,
which sends an HTTP request to DazScriptServer and updates the joint live.

> Change `"Agnes Montenegro"` and `"head"` to match a figure and bone in your scene.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# --- target -----------------------------------------------------------
FIGURE = "Agnes Montenegro"   # label shown in DAZ Studio Scene panel
BONE   = "head"               # bone name (case-sensitive)

figure = scene.find_skeleton_by_label(FIGURE)
bone   = figure.find_bone(BONE)

# --- sliders ----------------------------------------------------------
slider_x = widgets.FloatSlider(value=0, min=-90, max=90, step=0.5,
                               description="X (tilt)",
                               continuous_update=True, layout=widgets.Layout(width="450px"))
slider_y = widgets.FloatSlider(value=0, min=-90, max=90, step=0.5,
                               description="Y (turn)",
                               continuous_update=True, layout=widgets.Layout(width="450px"))
slider_z = widgets.FloatSlider(value=0, min=-90, max=90, step=0.5,
                               description="Z (roll)",
                               continuous_update=True, layout=widgets.Layout(width="450px"))

def _update(_change):
    bone.set_local_rotation(slider_x.value, slider_y.value, slider_z.value)

for s in (slider_x, slider_y, slider_z):
    s.observe(_update, names="value")

reset_btn = widgets.Button(description="Reset", button_style="warning")

def _reset(_btn):
    # Temporarily suppress callbacks so we only send one request
    for s in (slider_x, slider_y, slider_z):
        s.unobserve(_update, names="value")
        s.value = 0
        s.observe(_update, names="value")
    bone.set_local_rotation(0, 0, 0)

reset_btn.on_click(_reset)

display(widgets.VBox([
    widgets.HTML(f"<b>{FIGURE} — {BONE}</b>"),
    slider_x, slider_y, slider_z,
    reset_btn,
]))

---
## 9. Morph / property explorer

DAZ figures ship with hundreds of morphs (body shapes, expressions, correctives).
`figure.morph_values()` fetches them all in **one HTTP call**.

| Method | What it does |
|---|---|
| `figure.morph_values()` | `{name: float}` for every `DzMorph` modifier |
| `figure.morph_values(nonzero_only=True)` | Same, but skips values ≤ 0.0001 |
| `figure.set_morph_values({name: float, ...})` | Set any subset of morphs in one call |

> **Tip:** morph names are the internal `getName()` strings (e.g. `"FBMBreastSize"`,
> `"PHMSmileSimple"`), not the labels shown in the Parameters pane.
> Use the explorer cells below to discover them.

In [ ]:
# --- list all morphs on a figure -------------------------------------------
FIGURE = "Agnes Montenegro"  # change to match your scene

figure = scene.find_skeleton_by_label(FIGURE)
all_morphs = figure.morph_values()          # {name: float}, one HTTP call
active     = {k: v for k, v in all_morphs.items() if abs(v) > 0.0001}

print(f"{FIGURE}: {len(all_morphs)} morphs total, {len(active)} active")
print()

if active:
    print("Active morphs:")
    for name, val in sorted(active.items(), key=lambda x: -abs(x[1])):
        bar = '#' * int(abs(val) * 20)
        print(f"  {name:<40s}  {val:+.4f}  {bar}")
else:
    print("(all morphs are zero — showing first 20 available)")
    for name in list(all_morphs)[:20]:
        print(f"  {name}")


In [ ]:
# --- search morphs by keyword -----------------------------------------------
KEYWORD = "Smile"  # case-insensitive substring match

matches = {k: v for k, v in all_morphs.items() if KEYWORD.lower() in k.lower()}
print(f"{len(matches)} morphs matching {KEYWORD!r}:")
for name, val in sorted(matches.items()):
    print(f"  {name:<40s}  {val:+.4f}")


In [ ]:
# --- set morph values -------------------------------------------------------
# Only the morphs listed here are touched; all others stay as-is.
figure.set_morph_values({
    "PHMSmileSimple": 0.8,   # replace with names from your figure
    "FBMBreastSize":  0.3,
})
print("Morphs applied. Re-run the list cell to confirm.")


---
## 10. Scene tree pretty-printer

`scene.node_tree()` returns the full parent/child hierarchy as nested dicts, but the raw
JSON is hard to read with a complex scene. The helper below renders it as an indented tree.

Each line shows `label (internal_name)`. Nodes whose label matches their name are shown
with the name only to reduce noise.

In [ ]:
def print_tree(nodes, prefix="", last_flags=None):
    """Render a list of node dicts (from scene.node_tree()) as an indented tree."""
    if last_flags is None:
        last_flags = []
    for i, node in enumerate(nodes):
        is_last = i == len(nodes) - 1
        # Build the connector for this level
        connector = "\u2514\u2500\u2500 " if is_last else "\u251c\u2500\u2500 "
        # Build the indent from ancestor flags
        indent = "".join("    " if f else "\u2502   " for f in last_flags)
        name  = node["name"]
        label = node["label"]
        display = name if name == label else f"{label} ({name})"
        print(f"{indent}{connector}{display}")
        print_tree(node["children"], prefix, last_flags + [is_last])


tree = scene.node_tree()
print(f"Scene — {len(tree)} root node(s)")
print()
print_tree(tree)
